In [ ]:
# DATASET_WEBSITE = "https://www.unb.ca/cic/datasets/iotdataset-2023.html"

In [ ]:
from pathlib import Path
import pandas as pd
import os
import re
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# rekurencyjne szukanie wszystkich plików CSV w folderze i podfolderach
def iter_csv(dir_path):
    with os.scandir(dir_path) as a:
        for e in a:
            if e.is_file() and e.name.endswith(".csv"):
                yield e.path
            elif e.is_dir():
                yield from iter_csv(e.path)


# sprawdzanie ilości unikalncyh kolumn w plikach CSV (zwraca słownik)
def get_unique_columns(files_list):
    unique_sets = set{}
    
    for path in files_list:
        file_name = os.path.basename(path)
    
        try:
            df_header = pd.read_csv(path, nrows=0)
            columns = tuple(df_header.columns.str.strip())

            if columns not in unique_sets:
                unique_sets[columns] = []

            unique_sets[columns].append(path)
            
        except Exception as e:
            print(f"Błąd przy wczytywaniu pliku {file_name}: {e}")

    return unique_sets

# ujednolicanie tekstu dla kolumn i etykiet
def normalize_string(text):
    if isinstance(text, str):
        return re.sub(r'[\s_\-]+', '', text.lower())
    else:
        return None


In [ ]:
# tworzenie etykiet z nazw plików 
def get_label_from_filename(file_path):
    file_name = os.path.basename(file_path)

    label = file_name

    if label.endswith(".csv"):
        label = label[:-4]

    label = label.replace(".pcap", "")
    label = label.strip()

    return label
                     

In [ ]:
# etykieta binarna na podstawie nazwy pliku
def binary_label_from_filename(file_path):
    label = get_label_from_filename(file_path)
    normalized_label = normalize_string(label)

    if "benign" in normalized_label:
        return "benign"
    else:
        return "attack"
# podglad etykiet z nazw plików
def preview_file_labels(csv_files, max_files=20):
        for file_path in csv_files[:max_files]:
        file_name = os.path.basename(file_path)
        label = get_label_from_filename(file_path)
        binary_label = binary_label_from_filename(file_path)

        print(f"{file_name} ===> label: {label}, binary_label: {binary_label}")


In [ ]:
# dodawanie etykiet do chunka
def add_labels_from_filename(chunk, file_path):
    label = get_label_from_filename(file_path)
    binary_label = binary_label_from_filename(file_path)

    chunk["label"] = label
    chunk["binary_label"] = binary_label

    return chunk

In [ ]:
# rozklad klas binarnych, rozkład oryginalnych etykiet, liczba rekordów
def records_and_classes_count(csv_files, chunk_size=10000):
    total_records = 0
    original_label_distribution = {}
    binary_label_distribution = {}

    for file_path in csv_files:
        file_name = os.path.basename(file_path)

        label = get_label_from_filename(file_path)
        binary_label = binary_label_from_filename(file_path)

        try:
            for chunk in pd.read_csv(file_path, chunksize=chunk_size):
                records_in_chunk = len(chunk)
                total_records += records_in_chunk

                original_label_distribution[label] = (
                    original_label_distribution.get(label, 0) + records_in_chunk
                )

                binary_label_distribution[binary_label] = (
                    binary_label_distribution.get(binary_label, 0) + records_in_chunk
                )

        except Exception as e:
            print(f"Błąd przy przetwarzaniu pliku {file_name}: {e}")

    return total_records, original_label_distribution, binary_label_distribution


In [ ]:
# tworzenie stratyfikowanej próbki danych
def create_stratified_sample(csv_files, sample_proportion=0.05, chunk_size=10000, random_state=42):
        sampled_chunks = []

        for file_path in csv_files:
            file_name = os.path.basename(file_path)

            try:
                for chunk in pd.read_csv(file_path, chunksize=chunk_size):
                    chunk.columns = chunk.columns.str.strip()
                    chunk = add_labels_from_filename(chunk, file_path)
                        
                    sampled_chunk = chunk.sample(
                        frac=sample_proportion,
                        random_state=random_state
                    )
    
                    sampled_chunks.append(sampled_chunk)
    
            except Exception as e:
                print(f"Błąd przy próbkowaniu pliku {file_name}: {e}")

        if sampled_chunks:
            sample_df = pd.concat(sampled_chunks, ignore_index=True)
        else:
            sample_df = pd.DataFrame()
    
        return sample_df


In [ ]:
# wypisywanie rozkładu klas z procentami
def print_distribution(distribution, title):
    print(f"{title}")

    total = sum(distribution.values())

    for label, count in distribution.items():
        percent = 100 * count / total if total > 0 else 0
        print(f"{label}: {count} ({percent:.2f}%)")

In [ ]:
def main():
    data_dir = os.getcwd()

    csv_files = [file for file in iter_csv(data_dir)]
    print(f"Liczba plików CSV w folderze wynosi: {len(csv_files)}\n")

    if len(csv_files) == 0:
        print("Nie znaleziono plików CSV.")
        return

    unique_columns_results = get_unique_columns(csv_files)

    if len(unique_columns_results) != 1:
        print(f"!! Uwaga !! Wykryto {len(unique_columns_results)} różne struktury kolumn!!")

        for i, (columns, files) in enumerate(unique_columns_results.items(), start=1):
            print(f"\nUkład kolumn nr {i}")
            print(f"Liczba kolumn: {len(columns)}")
            print(f"Przykładowy plik: {os.path.basename(files[0])}")
            print(f"Kolumny: {list(columns)}")

        print("\nNajpierw trzeba zdecydować, które pliki mają być użyte.")
        return

    else:
        common_columns = list(next(iter(unique_columns_results.keys())))

        print("Wszystkie pliki mają identyczne kolumny.")
        print(f"Kolumny w plikach to: {common_columns}")
        print(f"\nIlość kolumn w oryginalnych plikach: {len(common_columns)}")

    preview_file_labels(csv_files)

    total_records, original_distribution, binary_distribution = records_and_classes_count(
        csv_files=csv_files,
        chunk_size=10_000
    )

    print(f"\nLiczba rekordów w całym zbiorze: {total_records}")
    print(f"Liczba kolumn przed dodaniem etykiet: {len(common_columns)}")
    print(f"Liczba kolumn po dodaniu label i binary_label: {len(common_columns) + 2}")

    print_distribution(original_distribution, "Rozkład etykiet z nazw plików")
    print_distribution(binary_distribution, "Rozkład klas binarnych")

    sample_df = create_stratified_sample(
        csv_files=csv_files,
        sample_proportion=0.05,
        chunk_size=10_000,
        random_state=42
    )

    if len(sample_df) == 0:
        print("Nie utworzono próbki.")
        return

    output_file = "ciciot2023_sample.parquet"
    sample_df.to_parquet(output_file, index=False)

    print(f"\nZapisano próbkę do pliku: {output_file}")
    print(f"Liczba rekordów w próbce: {len(sample_df)}")
    print(f"Liczba kolumn w próbce: {len(sample_df.columns)}")

    sample_original_distribution = sample_df["label"].value_counts().to_dict()
    sample_binary_distribution = sample_df["binary_label"].value_counts().to_dict()

    print_distribution(sample_original_distribution, "Rozkład etykiet w próbce")
    print_distribution(sample_binary_distribution, "Rozkład klas binarnych w próbce")

#if __name__ == "__main__":
#   main()

In [ ]:
main()